In [0]:
# Load Configuration
# Create database & Tables
# Check Full Load / Delta Load
# Read Data Based on the Load Type
# Incrementally Processing data logic
# Write Data to Delta Table

**Load Depedency**

In [0]:
%run /Workspace/Users/dipanjannet@gmail.com/supplychain-etl-pipeline/src/transformation/comon/variables

In [0]:
%run /Workspace/Users/dipanjannet@gmail.com/supplychain-etl-pipeline/src/transformation/comon/Utility

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W

**Variable Declaration**

In [0]:
# Get input parameters from Databricks Job
source_system = dbutils.widgets.get("source_system")
source_table_name = dbutils.widgets.get("source_table_name")
key_col_name = dbutils.widgets.get("key_col_name")
target_table_name = dbutils.widgets.get("target_table_name")


# Constract the source table name
source_table_path = f"{BRONZE_DATA_LAKE_PATH}/{source_system}/{source_table_name}/*/*/*"

# Constract the target table name
target_database_name = SILVER_DATABASE_NAME + "_" + source_system
target_database_table_name = target_database_name + "." + target_table_name
target_table_path = f"{SILVER_DATA_LAKE_PATH}/{source_system}/{target_table_name}"

# Source Details
print("source_table_path: ", source_table_path)
# Target Details
print("target_database_name: ", target_database_name)
print("target_table_name: ", target_database_table_name)
print("target_table_path: ", target_table_path)

**Idepotant Silver Data Loadng**

In [0]:
full_load_flag = is_full_load(spark, target_database_name, target_table_name, target_table_path)
print(full_load_flag)

if full_load_flag:
  print("Performig Full Load...")
  df = spark.read.format("parquet").load(source_table_path)
else:
  # As a delta load, we need to get the max timestamp from the target table
  print("Performig Delta Load...")
  sql_query = f"select max(file_modification_time) from {target_database_table_name}"
  modified_after = spark.sql(sql_query).collect()[0][0]
  print("modified_after: ", modified_after)
  # Read data based on the modified_after timestamp
  df = spark.read.format("parquet").load(source_table_path)
  df = add_dataframe_metadata(df)
  df = df.filter(
    F.col("file_modification_time") > F.lit(modified_after)
  )

print("Reading data from source table completed")

if not df.isEmpty():
  print("Dataframe count: ", df.count())
  column_list = df.columns
  # We will Maintain History for All Tables
  extra_cols = [key_col_name] if isinstance(key_col_name, str) and key_col_name else []
  print("Adding MD5 Hash to the dataframe...")
  df = generate_md5_hash(df, extra_cols)
  # Add Audit Column
  df = df.withColumn("pz_created_ts", F.current_timestamp())

  # # Add Dataframe Metadata
  # df = add_dataframe_metadata(df)
  # print("Dataframe Metadata Added")

  # We also need to Deduplicate the Data before Merging
  extra_cols =  extra_cols + ["md5_hash"]
  print("Deduplicating Dataframe..", extra_cols)
  _w = W.partitionBy(extra_cols).orderBy(F.col("pz_created_ts").desc())
  df = df.withColumn("row_number", F.row_number().over(_w)).filter(F.col("row_number") == 1).drop("row_number")
  
  # Final Selection of Columns
  column_list = ["pz_created_ts", "ingest_file_name", "file_modification_time", "md5_hash"] + column_list
  df = df.select(*column_list)

  # Write Data to Silver lake
  if full_load_flag:
    print("Full Load! - Overwriting data")
    df.write.format("delta").mode("overwrite").option("path", target_table_path)\
    .saveAsTable(target_database_table_name)
  else:
    print("Incremental Load! - Appending data and Maintaing History")
    target_table = DeltaTable.forPath(spark, target_table_path)
    try:
      target_table.alias("t").merge(
        df.alias("s"),
        f"t.{key_col_name} = s.{key_col_name} AND t.md5_hash = s.md5_hash"
      ).whenNotMatchedInsertAll().execute()
    except Exception as e:
      print("Merge Opration Faild. Possible Reason : ", e)

    # df.write.format("delta").mode("append").option("path", target_table_path)\
    # .saveAsTable(target_database_table_name)
  
else:
  print("No new data to process| Skipping Ingestion...")